# Python 工程实践 — 高级工程师面试精讲

本笔记聚焦于生产级 Python 代码的工程质量：测试、类型系统、可观测性、性能诊断和包管理。

| 主题 | 出现频率 |
|------|----------|
| 单元测试 pytest / mock / fixture | 高频 |
| 类型注解 typing & mypy | 重要 |
| logging 模块最佳实践 | 重要 |
| cProfile / line_profiler 性能分析 | 重要 |
| tracemalloc 内存泄漏排查 | 重要 |
| packaging: pyproject.toml / poetry | 了解 |

---
## 1. pytest — 单元测试框架

### 为什么用 pytest 而非 unittest？

| 特性 | unittest | pytest |
|------|----------|--------|
| 语法 | 类继承 `TestCase` | 普通函数，assert 语句 |
| fixture | setUp/tearDown | 依赖注入，scope 可控 |
| 参数化 | 手动循环 | `@pytest.mark.parametrize` |
| 插件生态 | 少 | 丰富（pytest-cov, pytest-asyncio...）|

### Fixture 的 scope 层级

| scope | 执行次数 | 适用场景 |
|-------|---------|----------|
| `function`（默认）| 每个测试函数 | 轻量级对象 |
| `class` | 每个测试类 | 类级共享 |
| `module` | 每个测试文件 | DB 连接、文件 |
| `session` | 整个测试会话 | 全局配置、重型初始化 |

In [ ]:
# pytest must be run from terminal: pytest test_file.py -v
# In this notebook we demonstrate patterns and run tests inline using pytest.main()

# --- Install if needed ---
# !pip install pytest pytest-cov

import pytest
print(f"pytest version: {pytest.__version__}")

In [ ]:
# ============================================================
# Production code being tested
# ============================================================
from dataclasses import dataclass
from typing import Optional

@dataclass
class Order:
    order_id: str
    amount: float
    discount: float = 0.0
    tax_rate: float = 0.1

    def total(self) -> float:
        """Calculate total after discount and tax."""
        if not 0 <= self.discount <= 1:
            raise ValueError(f"Discount must be in [0, 1], got {self.discount}")
        discounted = self.amount * (1 - self.discount)
        return round(discounted * (1 + self.tax_rate), 2)

    def apply_coupon(self, code: str) -> 'Order':
        """Apply a coupon code, returning new Order with updated discount."""
        coupons = {"SAVE10": 0.10, "SAVE20": 0.20, "VIP50": 0.50}
        if code not in coupons:
            raise KeyError(f"Invalid coupon: {code}")
        return Order(self.order_id, self.amount, coupons[code], self.tax_rate)

# Quick smoke test
o = Order("O001", 100.0, 0.1, 0.1)
print(f"Order total: {o.total()}")  # (100 * 0.9) * 1.1 = 99.0

In [ ]:
%%writefile /tmp/test_order.py
"""
Demonstrates:
- fixtures with different scopes
- yield fixtures for setup/teardown
- parametrize
- testing exceptions
"""
import pytest
from dataclasses import dataclass

# ---- Inline production code (normally imported) ----
@dataclass
class Order:
    order_id: str
    amount: float
    discount: float = 0.0
    tax_rate: float = 0.1

    def total(self) -> float:
        if not 0 <= self.discount <= 1:
            raise ValueError(f"Discount must be in [0, 1], got {self.discount}")
        return round(self.amount * (1 - self.discount) * (1 + self.tax_rate), 2)

    def apply_coupon(self, code: str) -> 'Order':
        coupons = {"SAVE10": 0.10, "SAVE20": 0.20, "VIP50": 0.50}
        if code not in coupons:
            raise KeyError(f"Invalid coupon: {code}")
        return Order(self.order_id, self.amount, coupons[code], self.tax_rate)

# ---- Fixtures ----

@pytest.fixture
def basic_order():
    """Function-scoped fixture: fresh Order for each test."""
    return Order("O001", 100.0)

@pytest.fixture(scope="module")
def expensive_resource():
    """Module-scoped yield fixture: setup → yield → teardown."""
    print("\n[fixture] Setting up module-scoped resource")
    resource = {"db": "connected", "records": 1000}
    yield resource
    # Teardown runs after all tests in the module
    print("\n[fixture] Tearing down module-scoped resource")
    resource["db"] = "disconnected"

# ---- Tests ----

def test_order_total_no_discount(basic_order):
    assert basic_order.total() == 110.0  # 100 * 1.1

def test_order_total_with_discount(basic_order):
    basic_order.discount = 0.2
    assert basic_order.total() == 88.0   # 100 * 0.8 * 1.1

def test_module_resource(expensive_resource):
    assert expensive_resource["db"] == "connected"
    assert expensive_resource["records"] == 1000

@pytest.mark.parametrize("amount,discount,tax,expected", [
    (100.0, 0.0,  0.1,  110.00),
    (200.0, 0.5,  0.1,  110.00),
    (50.0,  0.1,  0.2,   54.00),
    (0.0,   0.0,  0.15,   0.00),
])
def test_order_total_parametrized(amount, discount, tax, expected):
    order = Order("test", amount, discount, tax)
    assert order.total() == expected

def test_invalid_discount_raises():
    order = Order("O999", 100.0, discount=1.5)  # invalid
    with pytest.raises(ValueError, match="Discount must be in"):
        order.total()

def test_invalid_coupon_raises(basic_order):
    with pytest.raises(KeyError, match="Invalid coupon"):
        basic_order.apply_coupon("INVALID")

def test_apply_coupon_returns_new_order(basic_order):
    new_order = basic_order.apply_coupon("SAVE10")
    assert new_order is not basic_order      # immutable: new object
    assert new_order.discount == 0.10
    assert basic_order.discount == 0.0       # original unchanged


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/tmp/test_order.py", "-v", "--tb=short", "-s"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

### 1.2 Mock & Patch

Mock 用于隔离**外部依赖**（数据库、HTTP、文件系统、时钟）：

| 工具 | 用途 |
|------|------|
| `MagicMock` | 自动创建 mock 对象，支持任意属性/方法调用 |
| `patch` | 在指定作用域内替换对象 |
| `patch.object` | 替换特定对象的属性 |
| `spec=` | 限制 mock 只能有真实对象的属性，防止测试假阳性 |

In [ ]:
%%writefile /tmp/test_mock.py
"""
Demonstrates:
- unittest.mock.patch as decorator and context manager
- MagicMock side_effect and return_value
- spec parameter to prevent false positives
- patching in the right namespace
"""
import pytest
from unittest.mock import MagicMock, patch, call
from datetime import datetime

# ---- Production code ----
class DatabaseClient:
    def __init__(self, dsn: str):
        self.dsn = dsn

    def query(self, sql: str) -> list:
        raise NotImplementedError("Real DB call")

    def execute(self, sql: str) -> int:
        raise NotImplementedError("Real DB call")

class UserService:
    def __init__(self, db: DatabaseClient):
        self.db = db

    def get_active_users(self) -> list:
        rows = self.db.query("SELECT id, name FROM users WHERE active = 1")
        return [{"id": r[0], "name": r[1]} for r in rows]

    def deactivate_user(self, user_id: int) -> bool:
        affected = self.db.execute(f"UPDATE users SET active=0 WHERE id={user_id}")
        return affected > 0

# ---- Tests with MagicMock ----

def test_get_active_users_returns_dict_list():
    mock_db = MagicMock(spec=DatabaseClient)  # spec prevents typo mistakes
    mock_db.query.return_value = [(1, "Alice"), (2, "Bob")]

    service = UserService(db=mock_db)
    users = service.get_active_users()

    assert users == [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]
    mock_db.query.assert_called_once_with("SELECT id, name FROM users WHERE active = 1")

def test_deactivate_user_returns_true_when_affected():
    mock_db = MagicMock(spec=DatabaseClient)
    mock_db.execute.return_value = 1  # 1 row affected

    service = UserService(db=mock_db)
    result = service.deactivate_user(42)

    assert result is True
    mock_db.execute.assert_called_once()

def test_deactivate_user_returns_false_when_not_found():
    mock_db = MagicMock(spec=DatabaseClient)
    mock_db.execute.return_value = 0  # 0 rows affected

    service = UserService(db=mock_db)
    assert service.deactivate_user(9999) is False

def test_db_exception_propagates():
    mock_db = MagicMock(spec=DatabaseClient)
    mock_db.query.side_effect = ConnectionError("DB connection failed")

    service = UserService(db=mock_db)
    with pytest.raises(ConnectionError, match="DB connection failed"):
        service.get_active_users()

@patch("datetime.datetime")  # patch in the namespace where it's used
def test_patch_datetime(mock_dt):
    mock_dt.now.return_value = datetime(2024, 1, 15, 12, 0, 0)
    now = datetime.now()
    assert now.year == 2024
    assert now.month == 1


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/tmp/test_mock.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

---
## 2. 类型注解 — typing & mypy

### 类型系统演进

| 版本 | 新增 |
|------|------|
| 3.5 | `typing` 模块：`List`, `Dict`, `Optional`, `Union` |
| 3.8 | `TypedDict`, `Literal`, `Protocol`, `Final` |
| 3.9 | 内置容器泛型：`list[int]` 代替 `List[int]` |
| 3.10 | `X | Y` 代替 `Union[X, Y]`，`match` 语句 |
| 3.11 | `Self`, `Never`, `TypeVarTuple` |
| 3.12 | `TypeAlias` 正式化，`override` 标注 |

### 核心类型工具

| 工具 | 用途 |
|------|------|
| `TypeVar` | 泛型函数/类型参数 |
| `Generic[T]` | 泛型类 |
| `Protocol` | 结构子类型（duck typing 的类型化）|
| `TypedDict` | 有类型约束的字典 |
| `Literal` | 限制值为特定字面量 |
| `overload` | 同一函数多种签名 |

In [ ]:
from __future__ import annotations
from typing import TypeVar, Generic, Protocol, TypedDict, Literal, overload, runtime_checkable
from collections.abc import Iterator, Sequence

# --- TypeVar & Generic ---
T = TypeVar("T")
E = TypeVar("E", bound=Exception)

class Stack(Generic[T]):
    """Type-safe stack container."""
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("Stack is empty")
        return self._items.pop()

    def peek(self) -> T:
        return self._items[-1]

    def __len__(self) -> int:
        return len(self._items)

# Usage
int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
print(f"Stack peek: {int_stack.peek()}, len: {len(int_stack)}")
print(f"Popped: {int_stack.pop()}")

In [ ]:
from typing import Protocol, runtime_checkable, TypedDict, Literal
from collections.abc import Sequence

# --- Protocol: structural subtyping (duck typing + type safety) ---
@runtime_checkable  # allows isinstance() checks at runtime
class Serializable(Protocol):
    """Any class with these methods satisfies this protocol."""
    def to_json(self) -> str: ...
    def to_dict(self) -> dict: ...

class User:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def to_json(self) -> str:
        import json
        return json.dumps(self.to_dict())

    def to_dict(self) -> dict:
        return {"name": self.name, "age": self.age}

# User never explicitly declares Serializable — structural subtyping
user = User("Alice", 30)
print(f"isinstance(user, Serializable): {isinstance(user, Serializable)}")
print(f"user.to_json(): {user.to_json()}")

def serialize_all(items: Sequence[Serializable]) -> list[str]:
    return [item.to_json() for item in items]

print(f"Serialize: {serialize_all([user])}")

print()

# --- TypedDict: typed structure for dict-like data ---
class PipelineConfig(TypedDict):
    name: str
    source_table: str
    batch_size: int
    schedule: Literal["hourly", "daily", "weekly"]  # only these 3 values

def run_pipeline(config: PipelineConfig) -> None:
    print(f"Running {config['name']} on schedule={config['schedule']}, batch={config['batch_size']}")

config: PipelineConfig = {
    "name": "sales_etl",
    "source_table": "raw.sales",
    "batch_size": 10000,
    "schedule": "daily",
}
run_pipeline(config)

In [ ]:
from typing import overload

# --- overload: multiple signatures for the same function ---
@overload
def process(data: str) -> str: ...
@overload
def process(data: list) -> list: ...
@overload
def process(data: dict) -> dict: ...

def process(data):
    """Actual implementation handles all types."""
    if isinstance(data, str):
        return data.strip().upper()
    elif isinstance(data, list):
        return [item for item in data if item is not None]
    elif isinstance(data, dict):
        return {k: v for k, v in data.items() if v is not None}
    else:
        raise TypeError(f"Unsupported type: {type(data)}")

print(process("  hello world  "))      # str → str
print(process([1, None, 2, None, 3])) # list → list
print(process({"a": 1, "b": None, "c": 3}))  # dict → dict

# --- ParamSpec: preserve function signatures in decorators (Python 3.10+) ---
from typing import ParamSpec, Callable
import functools

P = ParamSpec("P")
R = TypeVar("R")

def log_args(func: Callable[P, R]) -> Callable[P, R]:
    """Type-safe decorator that preserves the wrapped function's signature."""
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        print(f"[log] {func.__name__}({args!r}, {kwargs!r})")
        return func(*args, **kwargs)
    return wrapper

@log_args
def add(x: int, y: int, *, z: int = 0) -> int:
    return x + y + z

print(f"Result: {add(1, 2, z=3)}")

---
## 3. logging 模块最佳实践

### 关键原则

1. **永远不要用 `print` 代替 `logging`** — print 无法控制级别、无法路由到不同 handler
2. **在库代码中使用 `logging.getLogger(__name__)`** — 让调用方决定如何处理日志
3. **不要在库中调用 `basicConfig`** — 这会影响整个应用的日志配置
4. **生产环境使用结构化日志（JSON）** — 方便 ELK/Splunk 解析
5. **使用 `%s` 懒惰格式化，而非 f-string** — 未到达输出级别时不执行格式化

### Logger 层级

```
root
├── myapp
│   ├── myapp.pipeline
│   └── myapp.utils
└── third_party
```

日志从子 logger 向上传播，直到找到有 handler 的 logger。

In [ ]:
import logging
import sys

# --- Basic logger setup (application entry point) ---
def setup_logging(level: int = logging.INFO) -> None:
    """Configure root logger — call once at application startup."""
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)-8s | %(name)s:%(lineno)d | %(message)s",
        datefmt="%Y-%m-%dT%H:%M:%S",
        handlers=[
            logging.StreamHandler(sys.stdout),   # console
            # logging.FileHandler("app.log"),   # add file handler in production
        ]
    )

setup_logging(logging.DEBUG)

# --- Library code: use module-level logger ---
logger = logging.getLogger(__name__)  # __name__ = '__main__' in notebook

def process_batch(batch_id: str, records: int) -> None:
    logger.info("Starting batch %s with %d records", batch_id, records)

    try:
        if records <= 0:
            raise ValueError(f"records must be positive, got {records}")
        logger.debug("Batch %s processed successfully", batch_id)
    except ValueError:
        logger.exception("Failed to process batch %s", batch_id)  # logs full traceback
        raise

process_batch("B001", 1000)
try:
    process_batch("B002", -1)
except ValueError:
    pass

In [ ]:
import logging
import json
import sys
import traceback
from datetime import datetime, timezone

# --- Structured JSON logging (production-ready) ---
class JsonFormatter(logging.Formatter):
    """Outputs log records as JSON lines — compatible with ELK, Datadog, etc."""

    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level":     record.levelname,
            "logger":    record.name,
            "message":   record.getMessage(),
            "module":    record.module,
            "line":      record.lineno,
        }
        # Include extra fields passed via extra={}
        for key, value in record.__dict__.items():
            if key.startswith("x_"):  # convention: extra fields prefixed with x_
                log_entry[key] = value

        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)

        return json.dumps(log_entry, ensure_ascii=False)


# Set up JSON logger
json_logger = logging.getLogger("json_demo")
json_logger.setLevel(logging.DEBUG)
json_logger.handlers = []  # clear existing handlers

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JsonFormatter())
json_logger.addHandler(handler)
json_logger.propagate = False

# Usage with structured context
json_logger.info(
    "Pipeline completed",
    extra={"x_pipeline_id": "etl_v2", "x_records": 50000, "x_duration_ms": 1234}
)

try:
    1 / 0
except ZeroDivisionError:
    json_logger.exception(
        "Unexpected error in calculation",
        extra={"x_pipeline_id": "etl_v2", "x_step": "normalize"}
    )

---
## 4. 性能分析 — cProfile & line_profiler

### 分析流程

```
1. 先用 cProfile 找热点函数（哪个函数总耗时最多？）
2. 对热点函数用 line_profiler 定位具体行
3. 优化，再次测量验证
```

### cProfile 关键指标

| 指标 | 含义 |
|------|------|
| `ncalls` | 调用次数 |
| `tottime` | 函数本身耗时（不含子函数）|
| `cumtime` | 累计耗时（含子函数）|
| `percall` | 每次调用平均耗时 |

**排序建议**：先按 `cumtime` 排序找顶层热点，再按 `tottime` 找底层瓶颈。

In [ ]:
import cProfile
import pstats
import io
import re

# --- Profiling target: intentionally suboptimal code ---
def slow_word_count(text: str) -> dict:
    """Counts word frequencies — uses apply-style loop."""
    words = text.lower().split()
    counts = {}
    for word in words:
        word = re.sub(r'[^a-z]', '', word)  # slow: regex in loop
        if word:
            counts[word] = counts.get(word, 0) + 1
    return counts

def fast_word_count(text: str) -> dict:
    """Counts word frequencies — vectorized approach."""
    from collections import Counter
    # Compile regex once, outside loop
    words = re.sub(r'[^a-z\s]', '', text.lower()).split()
    return Counter(words)

# Generate test data
import random, string
random.seed(42)
vocab = ["data", "engineer", "pipeline", "spark", "kafka", "python", "sql", "cloud"]
test_text = " ".join(random.choices(vocab, k=50_000))

# --- Profile slow version ---
profiler = cProfile.Profile()
profiler.enable()
slow_result = slow_word_count(test_text)
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("cumulative")
stats.print_stats(10)  # top 10 functions by cumtime
print("=== cProfile output (slow_word_count) ===")
print(stream.getvalue())

In [ ]:
import time

# --- Compare slow vs fast ---
N_RUNS = 5

start = time.perf_counter()
for _ in range(N_RUNS):
    slow_word_count(test_text)
slow_time = (time.perf_counter() - start) / N_RUNS

start = time.perf_counter()
for _ in range(N_RUNS):
    fast_word_count(test_text)
fast_time = (time.perf_counter() - start) / N_RUNS

print(f"slow_word_count: {slow_time*1000:.1f} ms/call")
print(f"fast_word_count: {fast_time*1000:.1f} ms/call")
print(f"Speedup: {slow_time/fast_time:.1f}x")

# Verify same results
assert slow_word_count(test_text) == dict(fast_word_count(test_text)), "Results differ!"
print("Results match: OK")

print()
print("=== Profiling Tips ===")
print("1. Use cProfile to find HOT functions (sort by cumtime)")
print("2. Use line_profiler (@profile) to find HOT lines within a function")
print("3. Install: pip install line-profiler")
print("4. Run:     kernprof -l -v script.py")
print("5. Visualize cProfile with: pip install snakeviz && snakeviz profile.prof")

In [ ]:
import cProfile
import pstats
import io
from contextlib import contextmanager

# --- Reusable profiling context manager ---
@contextmanager
def profile(sort_by: str = "cumulative", top_n: int = 10):
    """Context manager for easy ad-hoc profiling."""
    pr = cProfile.Profile()
    pr.enable()
    try:
        yield pr
    finally:
        pr.disable()
        stream = io.StringIO()
        ps = pstats.Stats(pr, stream=stream)
        ps.sort_stats(sort_by)
        ps.print_stats(top_n)
        # Filter out built-in noise for readability
        output = "\n".join(
            line for line in stream.getvalue().splitlines()
            if line.strip() and "frozen" not in line
        )
        print(output)

# Usage
import numpy as np

with profile(sort_by="tottime", top_n=5):
    # Profile block of code
    arr = np.random.randn(1000, 1000)
    result = np.linalg.svd(arr)
    _ = arr @ arr.T

---
## 5. 内存泄漏排查 — tracemalloc

### tracemalloc 工作原理

Python 的 `tracemalloc` 模块跟踪内存分配的**调用栈**，让你找到是哪行代码分配了最多内存。

### 典型内存泄漏场景

| 场景 | 原因 |
|------|------|
| 无限增长的全局缓存 | 未设置大小限制 |
| 循环引用 + `__del__` | 旧版 Python 的 GC 盲区 |
| 未关闭的文件/连接 | `with` 语句或 `try/finally` 缺失 |
| Event listener 累积 | 注册了 listener 但未注销 |
| 大对象留在 local scope | 函数执行完未释放（被闭包引用）|

In [ ]:
import tracemalloc
import linecache

def display_top_allocations(snapshot, limit: int = 5):
    """Display top memory-consuming lines from a tracemalloc snapshot."""
    stats = snapshot.statistics("lineno")
    print(f"\nTop {limit} memory allocations:")
    print("-" * 70)
    for stat in stats[:limit]:
        frame = stat.traceback[0]
        print(f"{stat.size / 1024:.1f} KiB | {frame.filename}:{frame.lineno}")
        line = linecache.getline(frame.filename, frame.lineno).strip()
        if line:
            print(f"  → {line}")
    print("-" * 70)

# --- Detect memory leak: unbounded cache ---
leaky_cache = {}   # global cache with no size limit

def leaky_function(key: str) -> bytes:
    """Caches results but never evicts — memory leak."""
    if key not in leaky_cache:
        leaky_cache[key] = b"x" * 10_000   # 10 KB per entry
    return leaky_cache[key]

# Take snapshot BEFORE
tracemalloc.start()
snapshot_before = tracemalloc.take_snapshot()

# Simulate many unique calls
for i in range(500):
    leaky_function(f"key_{i}")

# Take snapshot AFTER
snapshot_after = tracemalloc.take_snapshot()
tracemalloc.stop()

# Compare snapshots
stats = snapshot_after.compare_to(snapshot_before, "lineno")
print("Memory growth since snapshot_before:")
print("-" * 70)
for stat in stats[:5]:
    if stat.size_diff > 0:
        frame = stat.traceback[0]
        print(f"+{stat.size_diff/1024:.1f} KiB | {frame.filename}:{frame.lineno}")

print(f"\nCache grew to {len(leaky_cache)} entries = ~{len(leaky_cache)*10:.0f} KB")

In [ ]:
import tracemalloc
from functools import lru_cache

# --- Fix: bounded cache with lru_cache ---
@lru_cache(maxsize=100)  # evicts LRU entries after 100 items
def cached_compute(key: int) -> bytes:
    return b"x" * 10_000

tracemalloc.start()
snap1 = tracemalloc.take_snapshot()

for i in range(500):
    cached_compute(i % 200)  # 200 unique keys, maxsize=100

snap2 = tracemalloc.take_snapshot()
tracemalloc.stop()

total_growth = sum(s.size_diff for s in snap2.compare_to(snap1, "lineno") if s.size_diff > 0)
print(f"lru_cache version memory growth: {total_growth / 1024:.1f} KiB")
print(f"Cache info: {cached_compute.cache_info()}")

print()
print("=== Memory Leak Checklist ===")
checklist = [
    "Use lru_cache(maxsize=N) or cachetools for bounded caching",
    "Always use 'with' statements for files, connections, locks",
    "Use weakref for observer/listener patterns",
    "Profile memory periodically in long-running services",
    "tracemalloc.take_snapshot() + compare_to() to find growth",
]
for item in checklist:
    print(f"  - {item}")

---
## 6. Packaging — pyproject.toml & Poetry

### 现代 Python 包管理演进

```
setup.py (旧) → setup.cfg (过渡) → pyproject.toml (现代标准, PEP 517/518/621)
```

### pyproject.toml 结构

- `[build-system]`：指定构建后端（poetry-core, hatchling, flit-core, setuptools）
- `[project]`：包元数据（PEP 621 标准）
- `[project.optional-dependencies]`：extras（如 `pip install mypackage[dev]`）
- `[tool.pytest.ini_options]`：pytest 配置
- `[tool.mypy]`：mypy 配置
- `[tool.ruff]`：linter 配置

In [ ]:
# This cell shows a complete pyproject.toml for a data engineering package
# (displayed as a string since we can't actually write it in this notebook context)

pyproject_toml_example = '''
[build-system]
requires      = ["poetry-core>=1.8.0"]
build-backend = "poetry.core.masonry.api"

[tool.poetry]
name        = "de-pipeline-toolkit"
version     = "1.2.0"
description = "Data engineering pipeline utilities"
authors     = ["Your Name <you@example.com>"]
readme      = "README.md"
license     = "Apache-2.0"
packages    = [{include = "de_pipeline"}]

[tool.poetry.dependencies]
python   = ">=3.11,<4.0"
pandas   = ">=2.1"
pyarrow  = ">=14.0"
pydantic = ">=2.0"

[tool.poetry.group.dev.dependencies]
pytest       = ">=7.4"
pytest-cov   = ">=4.1"
mypy         = ">=1.5"
ruff         = ">=0.1"
polars       = ">=0.19"

[tool.poetry.scripts]
de-run = "de_pipeline.cli:main"

# ── pytest configuration ─────────────────────────────────────────────────────
[tool.pytest.ini_options]
testpaths       = ["tests"]
addopts         = "-v --tb=short --strict-markers"
markers         = [
    "slow: marks tests as slow (use -m 'not slow' to skip)",
    "integration: marks integration tests requiring live services",
]
filterwarnings  = ["error", "ignore::DeprecationWarning"]

# ── mypy configuration ───────────────────────────────────────────────────────
[tool.mypy]
python_version          = "3.11"
strict                  = true      # enables all strict checks
ignore_missing_imports  = true      # for packages without stubs
warn_return_any         = true
disallow_untyped_defs   = true

# ── ruff linter configuration ────────────────────────────────────────────────
[tool.ruff]
target-version  = "py311"
line-length     = 100
select          = ["E", "F", "I", "N", "UP", "ANN", "B", "SIM"]
ignore          = ["ANN101", "ANN102"]

[tool.ruff.per-file-ignores]
"tests/**" = ["ANN"]   # no type annotations required in tests
'''

print(pyproject_toml_example)

In [ ]:
# Key Poetry commands reference
poetry_commands = {
    "poetry new my-package": "Scaffold a new package with src layout",
    "poetry install": "Install all deps from poetry.lock (reproducible)",
    "poetry install --with dev": "Include dev dependencies",
    "poetry add pandas>=2.0": "Add a dependency (updates pyproject.toml + lock)",
    "poetry add --group dev pytest": "Add to dev group",
    "poetry remove requests": "Remove a dependency",
    "poetry update": "Update all deps within version constraints",
    "poetry lock --no-update": "Regenerate lock file without updating versions",
    "poetry run pytest": "Run command in project virtualenv",
    "poetry build": "Build wheel + sdist",
    "poetry publish": "Upload to PyPI",
    "poetry export -f requirements.txt": "Export deps for Docker/non-poetry envs",
}

print("=== Poetry Command Reference ===")
for cmd, desc in poetry_commands.items():
    print(f"  {cmd:<45} # {desc}")

print()
print("=== Project Structure (src layout) ===")
structure = """
my-package/
├── pyproject.toml
├── poetry.lock         ← commit this for reproducible builds
├── README.md
├── src/
│   └── de_pipeline/
│       ├── __init__.py
│       ├── pipeline.py
│       └── utils.py
└── tests/
    ├── conftest.py     ← shared fixtures
    ├── unit/
    └── integration/
"""
print(structure)

---
## 复习要点

### pytest
- Fixture 四种 scope：function（默认）< class < module < session
- `yield` fixture：yield 之前是 setup，yield 之后是 teardown（相当于 `__enter__/__exit__`）
- `@pytest.mark.parametrize` 一次测试多组输入
- `pytest.raises(ExcType, match=pattern)` 验证异常类型和消息
- 测试文件和函数名以 `test_` 开头

### Mock
- `MagicMock(spec=RealClass)` 限制 mock 只有真实类的属性（推荐）
- `patch("module.ClassName")` 替换指定命名空间中的对象
- `return_value` 设置返回值，`side_effect` 设置异常或多次返回值
- `mock.assert_called_once_with(args)` 验证调用

### typing
- `TypeVar` + `Generic[T]` 实现类型安全的容器
- `Protocol` 实现结构子类型（不需要继承）
- `TypedDict` 为字典提供类型约束
- `Literal["a","b"]` 限制值为特定字面量
- `overload` 声明同一函数的多种签名
- `ParamSpec` 保留装饰器的函数签名

### logging
- 库中用 `getLogger(__name__)`，不要用 `basicConfig`
- 用 `%s` 懒格式化，避免 f-string（未达到级别时不执行格式化）
- `logger.exception(msg)` 自动附加 traceback
- 生产环境用 JSON formatter，便于日志系统解析
- Handler 层级：StreamHandler（控制台）、FileHandler（文件）、RotatingFileHandler（日志轮转）

### cProfile
- `cProfile.Profile()` 程序化使用，`pstats.Stats` 排序输出
- 先按 `cumtime` 找调用链热点，再按 `tottime` 找底层函数
- `line_profiler` 逐行分析（需 `kernprof -l -v script.py`）
- `snakeviz` 可视化 `.prof` 文件（`pip install snakeviz`）

### tracemalloc
- `tracemalloc.start()` → 运行代码 → `take_snapshot()` → `compare_to()`
- `statistics("lineno")` 按行分组，`statistics("traceback")` 按调用栈分组
- 常见泄漏：无界缓存、未关闭资源、循环引用 + `__del__`
- 修复：`lru_cache(maxsize=N)`、`with` 语句、`weakref`

### pyproject.toml
- 现代标准（PEP 517/518/621），替代 `setup.py` + `requirements.txt`
- `[tool.pytest.ini_options]`、`[tool.mypy]`、`[tool.ruff]` 集中配置
- `poetry.lock` 必须提交到版本控制（保证构建可复现）
- `poetry export -f requirements.txt` 用于 Docker 构建

---
## 练习

### 练习 1 — pytest fixtures & parametrize

给定下面的 `DataValidator` 类，编写完整的 pytest 测试套件：

1. 创建一个 `module`-scoped fixture，模拟一个共享的配置字典
2. 创建一个 `yield` fixture，在每个测试前创建临时 CSV 文件，测试后删除
3. 用 `@pytest.mark.parametrize` 测试 `validate_age` 的边界值（0、17、18、120、121）
4. 测试 `validate_email` 对各种合法/非法格式的处理
5. 测试异常路径，确保传入 None 时抛出 `TypeError`

In [ ]:
# Production code to test
import re
from typing import Optional

class DataValidator:
    EMAIL_PATTERN = re.compile(r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$')

    def validate_age(self, age: Optional[int]) -> bool:
        if age is None:
            raise TypeError("age cannot be None")
        return 18 <= age <= 120

    def validate_email(self, email: Optional[str]) -> bool:
        if email is None:
            raise TypeError("email cannot be None")
        return bool(self.EMAIL_PATTERN.match(email))

    def validate_record(self, record: dict) -> list[str]:
        """Returns list of validation error messages (empty = valid)."""
        errors = []
        try:
            if not self.validate_age(record.get("age")):
                errors.append(f"Invalid age: {record.get('age')}")
        except TypeError as e:
            errors.append(str(e))
        try:
            if not self.validate_email(record.get("email")):
                errors.append(f"Invalid email: {record.get('email')}")
        except TypeError as e:
            errors.append(str(e))
        return errors

# TODO: Write the test file below (use %%writefile /tmp/test_validator.py)
# Then run with subprocess or pytest


### 练习 2 — Mock 外部依赖

给定下面的 `S3DataLoader` 类（依赖 boto3），在**不实际连接 AWS** 的情况下编写完整测试：

1. 用 `MagicMock(spec=...)` 模拟 `boto3.client('s3')`
2. 测试 `load_csv` 正常路径：mock 返回正确的 CSV 内容
3. 测试 `load_csv` 文件不存在时的异常处理（mock 抛出 `ClientError`）
4. 用 `patch` 测试 `list_files` 方法
5. 验证 mock 的调用次数和调用参数

In [ ]:
# Production code
import io
from typing import Optional
import pandas as pd

class S3DataLoader:
    def __init__(self, s3_client, bucket: str):
        self.s3 = s3_client
        self.bucket = bucket

    def load_csv(self, key: str) -> pd.DataFrame:
        """Download a CSV from S3 and return as DataFrame."""
        response = self.s3.get_object(Bucket=self.bucket, Key=key)
        content = response["Body"].read().decode("utf-8")
        return pd.read_csv(io.StringIO(content))

    def list_files(self, prefix: str = "") -> list[str]:
        """List all object keys in the bucket with given prefix."""
        response = self.s3.list_objects_v2(Bucket=self.bucket, Prefix=prefix)
        return [obj["Key"] for obj in response.get("Contents", [])]

# TODO: Write tests using MagicMock and patch
# Hint for mocking the response Body:
# mock_body = MagicMock()
# mock_body.read.return_value = b"col1,col2\n1,2\n3,4"
# mock_s3.get_object.return_value = {"Body": mock_body}


### 练习 3 — 类型注解与 Protocol

**任务**：为一个数据管道框架设计类型系统：

1. 定义 `Extractor[T]` Protocol：有 `extract() -> Iterator[T]` 方法
2. 定义 `Transformer[T, U]` Protocol：有 `transform(item: T) -> U` 方法
3. 定义 `Loader[U]` Protocol：有 `load(items: Sequence[U]) -> int` 方法（返回成功加载数）
4. 实现泛型函数 `run_pipeline(extractor, transformer, loader)` 串联三个阶段
5. 实现具体的 `CsvExtractor`、`UpperCaseTransformer`、`PrintLoader` 满足上述 Protocol
6. 确保代码可以通过 mypy 类型检查（注释说明关键类型推断）

In [ ]:
from typing import TypeVar, Protocol, Generic, Iterator
from collections.abc import Sequence

T = TypeVar("T")
U = TypeVar("U")

# TODO: 1. Define Extractor Protocol
class Extractor(Protocol[T]):
    def extract(self) -> Iterator[T]: ...

# TODO: 2. Define Transformer Protocol
# class Transformer(Protocol[T, U]): ...

# TODO: 3. Define Loader Protocol
# class Loader(Protocol[U]): ...

# TODO: 4. Implement run_pipeline
def run_pipeline(extractor, transformer, loader) -> int:
    pass

# TODO: 5. Implement concrete classes
class CsvExtractor:
    """Reads lines from an in-memory CSV string."""
    def __init__(self, csv_data: str):
        self.rows = csv_data.strip().splitlines()[1:]  # skip header

    def extract(self) -> Iterator[str]:
        yield from self.rows

# class UpperCaseTransformer: ...
# class PrintLoader: ...

# Test
# csv = "name\nalice\nbob\ncharlie"
# result = run_pipeline(CsvExtractor(csv), UpperCaseTransformer(), PrintLoader())
# print(f"Loaded {result} records")


### 练习 4 — 性能分析

**任务**：下面有两个实现相同功能的函数，使用 `cProfile` 和手动计时来找出性能瓶颈：

1. 用 `cProfile` + `pstats` 分析 `slow_pipeline` 的热点函数
2. 解释为什么 `slow_pipeline` 慢
3. 实现 `fast_pipeline` 改进版本（使用向量化操作）
4. 比较两者在 50,000 行数据上的耗时

In [ ]:
import pandas as pd
import numpy as np
import cProfile, pstats, io

np.random.seed(0)
N = 50_000
df = pd.DataFrame({
    "name":   [f"user_{i}" for i in range(N)],
    "score":  np.random.uniform(0, 100, N),
    "region": np.random.choice(["north","south","east","west"], N),
    "active": np.random.choice([True, False], N),
})

def slow_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """Process active users: normalize score, add grade label."""
    result = []
    for _, row in df.iterrows():  # row-wise iteration — very slow
        if row["active"]:
            normalized = (row["score"] - 50) / 50
            if normalized > 0.5:
                grade = "A"
            elif normalized > 0:
                grade = "B"
            else:
                grade = "C"
            result.append({
                "name":       row["name"],
                "normalized": round(normalized, 4),
                "grade":      grade,
                "region":     row["region"],
            })
    return pd.DataFrame(result)

# TODO: Profile slow_pipeline with cProfile

# TODO: Implement fast_pipeline using vectorized pandas operations
def fast_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    pass

# TODO: Compare timings and verify results match


### 练习 5 — 内存泄漏排查 + 结构化日志

**任务**：综合练习

**Part A（内存）**：下面的 `EventBus` 有内存泄漏，使用 `tracemalloc` 找出并修复：
- 注册了 1000 个 listener，但没有清理机制
- 修复方案：使用 `weakref` 或添加 `unsubscribe` 方法
- 验证修复后内存不再增长

**Part B（日志）**：为 `EventBus` 添加结构化日志：
- 每次 `publish` 记录 event_type, listener_count, 耗时（ms）
- 每次 `subscribe` 记录 subscriber 信息
- 使用 JSON formatter，包含 `x_component: "event_bus"` 字段

In [ ]:
import tracemalloc
import logging
from collections import defaultdict
from typing import Callable

# Leaky version
class EventBus:
    """Simple event bus — has a memory leak!"""
    def __init__(self):
        self._listeners: dict[str, list[Callable]] = defaultdict(list)

    def subscribe(self, event_type: str, callback: Callable) -> None:
        self._listeners[event_type].append(callback)  # keeps strong reference
        # BUG: callbacks accumulate forever — no removal mechanism

    def publish(self, event_type: str, data: dict) -> None:
        for callback in self._listeners.get(event_type, []):
            callback(data)

# Part A: Detect the leak
tracemalloc.start()
snap1 = tracemalloc.take_snapshot()

bus = EventBus()
listeners = []  # keep references to prevent GC
for i in range(1000):
    # Lambda creates a closure — holds memory
    cb = lambda data, i=i: f"listener_{i}: {data}"
    listeners.append(cb)
    bus.subscribe("user_signup", cb)

snap2 = tracemalloc.take_snapshot()
tracemalloc.stop()

growth = sum(s.size_diff for s in snap2.compare_to(snap1, "lineno") if s.size_diff > 0)
print(f"Memory growth after 1000 subscriptions: {growth/1024:.1f} KiB")
print(f"Total listeners registered: {len(bus._listeners.get('user_signup', []))}")

# TODO Part A: Fix the memory leak (use weakref or add unsubscribe)

# TODO Part B: Add structured JSON logging to EventBus
